# LoadData — xuất dataset mắt YOLO (open + closed)

**Lỗi cũ:** chỉ xuất `closed_eyes` và gán `class_id = 0` trong khi `dataset.yaml` quy ước `0 = open eye`, `1 = close eye`.

**Sửa:** xuất cả hai lớp, class khớp `dataset_eye.yaml`. Có thể chạy script:
`python prepare_eye_dataset.py --out datasets/eye --max-samples 8000`

In [ ]:
from datasets import DatasetDict, load_dataset

raw = load_dataset("MichalMlodawski/closed-open-eyes")
full = raw["train"]
split = full.train_test_split(test_size=0.2, seed=42)
rest = split["test"].train_test_split(test_size=0.5, seed=42)
dataset_split = DatasetDict({
    "train": split["train"],
    "val": rest["train"],
    "test": rest["test"],
})
print("train/val/test:", len(dataset_split["train"]), len(dataset_split["val"]), len(dataset_split["test"]))

In [ ]:
import os

OPEN = {"open", "open_eyes", "opened", "open eye", "awake", "0"}
CLOSED = {"closed", "closed_eyes", "close", "close eye", "close_eyes", "drowsy", "1"}

def coco_to_yolo(x, y, w, h, width, height):
    return [((2 * x + w) / (2 * width)), ((2 * y + h) / (2 * height)), w / width, h / height]

def class_of(label):
    name = str(label).strip().lower().replace("-", "_")
    if name in OPEN:
        return 0, "open"
    if name in CLOSED:
        return 1, "close"
    return None, None

def preprocessing(partition, data, out_root="datasets/eye", max_samples=8000):
    img_dir = f"{out_root}/images/{partition}"
    lab_dir = f"{out_root}/labels/{partition}"
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lab_dir, exist_ok=True)
    subset = data[partition]
    n = min(len(subset), max_samples)
    n_open = n_close = 0
    for i, sample in enumerate(subset.select(range(n))):
        class_id, tag = class_of(sample["Label"])
        if class_id is None:
            continue
        img = sample["Image_data"]["file"]
        width, height = int(img.size[0]), int(img.size[1])
        lines = []
        for box in (sample["Left_eye_react"], sample["Right_eye_react"]):
            xc, yc, w, h = coco_to_yolo(box[0], box[1], box[2], box[3], width, height)
            lines.append(f"{class_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
        stem = f"{i:06d}_{tag}"
        with open(f"{lab_dir}/{stem}.txt", "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
        img.save(f"{img_dir}/{stem}.png")
        if class_id == 0:
            n_open += 1
        else:
            n_close += 1
    print(f"{partition}: open={n_open} closed={n_close}")

preprocessing("train", dataset_split)
preprocessing("val", dataset_split)
preprocessing("test", dataset_split)